## Stage 1.2 — Translate ESCO Skills to Russian (OpenAI Batch API)

**Role:** Interim preprocessing step between Stage 1 and Stage 2. Translates all ~13,900 ESCO skill entries from English to Russian using the OpenAI Batch API, producing `skills_ru.csv` required by the Stage 2 multilingual skill extractor.

**Why this step is necessary:** The ESCO taxonomy does not natively include Russian-language skill labels. Since a significant share of Ukrainian job vacancies are written in Russian, without this translation the Stage 2 skill extractor would fall back to English labels for all Russian-language records, significantly reducing extraction quality.

> **This step has already been completed.** The output file `skills_ru.csv` is included in `data/stage_01_2/processed/`. You do not need to rerun this notebook unless you want to improve or redo the translation. Rerunning requires an OpenAI API key and may take up to 24 hours.

**Position:** Run after `stage_1_read_initial_data_fast.ipynb` and before any Stage 2 notebook.

# 1.2. Preprocessing Step (Stage 1.2): Translating ESCO Skills to Russian

## 1.2.1. Description

Prior to performing multilingual skill extraction, the ESCO skill dataset was enriched with Russian-language labels. This step was essential, as the original ESCO dataset includes skill descriptions in multiple European languages (e.g., English, German, French, Polish), but does not natively support Russian, despite Russian being one of the commonly used languages in Ukrainian job advertisements.

## 1.2.2. Packages and configuration

Cleaning a memory of Jupyter Notebook before run:

Load autoreload extension and import shared configuration. `clean_memory()` releases leftover variables from previous runs. `Config` provides the OpenAI API key from `.env`.

In [1]:
import os.path
%load_ext autoreload
%autoreload 2
import sys
sys.path.append("../code")
from general import clean_memory
clean_memory()
from general import Config

Import the OpenAI client library and standard utilities for JSON serialisation, CSV parsing, path handling, and text wrapping.

In [2]:
import json, csv, pathlib, textwrap
import openai

Define path constants and configuration for the translation job.

- `MODEL` — OpenAI model used for translation (`gpt-4.1-mini`)
- `SRC_CSV` — source ESCO skills in English (~13,900 rows)
- `BATCH_INPUT` — JSONL file to be created and uploaded to the Batch API
- `FUNC_SCHEMA_JSON` — function-calling schema enforcing the output structure
- `DST_CSV` — final output file with Russian translations

> **Note on paths:** If rerunning from this repository, update these paths to `../data/stage_01_2/processed/` and `../data/stage_01_2/intermediate/`.

In [10]:
MODEL            = "gpt-4.1-mini"
SRC_CSV          = pathlib.Path("../data/stage1_2/skills_en.csv")
BATCH_INPUT      = pathlib.Path("../data/stage1_2/batch_translate_en_ru.jsonl")
FUNC_SCHEMA_JSON = pathlib.Path("../data/stage1_2/translate_schema.json")
BATCH_POLL_SEC   = 20
DST_CSV          = pathlib.Path("../data/stage1_2/skills_ru.csv")

Set the OpenAI API key from the `.env` configuration. Required for all Batch API calls below.

In [11]:
openai.api_key = Config.OPENAI_API_KEY

Define and write the OpenAI function-calling schema to disk. The schema enforces that each API response contains exactly four fields: `conceptUri`, `preferredLabel_ru`, `altLabels_ru`, and `description_ru`. Function-calling guarantees structured, parseable JSON output for every skill.

In [12]:
# %% 1 ── write function schema file once ─────────────────────────────
schema = {
    "name": "translateSkillRecord",
    "description": "Translate ESCO skill fields from English to Russian.",
    "parameters": {
        "type": "object",
        "properties": {
            "conceptUri":        { "type": "string" },
            "preferredLabel_ru": { "type": "string" },
            "altLabels_ru":      { "type": "string" },
            "description_ru":    { "type": "string" }
        },
        "required": [
            "conceptUri",
            "preferredLabel_ru",
            "altLabels_ru",
            "description_ru"
        ]
    }
}
FUNC_SCHEMA_JSON.write_text(json.dumps([schema], ensure_ascii=False, indent=2))
print("✔ wrote", FUNC_SCHEMA_JSON)

✔ wrote ..\data\stage12\translate_schema.json


Define the system prompt sent to the model with every translation request. Instructions preserve technical jargon (e.g. C++, CI/CD) and the pipe separator used in `altLabels` fields, ensuring the translated CSV remains parseable.

In [13]:
sys_prompt = textwrap.dedent("""
    You are a professional translator.
    Translate every field from English to Russian, preserving punctuation,
    technical jargon (C++, JavaScript, CI/CD) and the pipe "|" separator
    in altLabels. Return ONLY a JSON object that matches the function schema.
""").strip()

## 1.2.3. Creating batch file

Define path constants and configuration for the translation job.

- `MODEL` — OpenAI model used for translation (`gpt-4.1-mini`)
- `SRC_CSV` — source ESCO skills in English (~13,900 rows)
- `BATCH_INPUT` — JSONL file to be created and uploaded to the Batch API
- `FUNC_SCHEMA_JSON` — function-calling schema enforcing the output structure
- `DST_CSV` — final output file with Russian translations

> **Note on paths:** If rerunning from this repository, update these paths to `../data/stage_01_2/processed/` and `../data/stage_01_2/intermediate/`.

In [14]:
with SRC_CSV.open(encoding="utf-8") as fin, BATCH_INPUT.open("w", encoding="utf-8") as fout:
    rdr = csv.DictReader(fin)
    for row in rdr:
        payload = {
            "conceptUri":     row["conceptUri"],
            "preferredLabel": row["preferredLabel"],
            "altLabels":      row["altLabels"],
            "description":    row["description"],
        }
        fout.write(json.dumps({
            "custom_id": row["conceptUri"],
            "method": "POST",
            "url": "/v1/chat/completions",
            "body": {
                "temperature": 0,
                "model": MODEL,
                "response_format": { "type": "json_object" },
                "functions": [ schema ],
                "messages": [
                    { "role": "system", "content": sys_prompt },
                    { "role": "user",   "content": json.dumps(payload, ensure_ascii=False) }
                ],
                "function_call": { "name": "translateSkillRecord" }
            }
        }, ensure_ascii=False) + "\n")
print("✔ built batch input", BATCH_INPUT)

✔ built batch input ..\data\stage12\batch_translate_en_ru.jsonl


Upload the JSONL batch input file to OpenAI. Returns a file object whose `.id` is needed to create the batch job in the next step.

In [ ]:
batch_file = openai.files.create(
    file=open(BATCH_INPUT, "rb"),
    purpose="batch")

## 1.2.4. Create batch job

Submit the translation batch job to the OpenAI Batch API with a 24-hour completion window.

> **After running this cell, stop and wait** for the batch job to complete before proceeding. Check status using the polling cell below.

In [ ]:
batch_job = openai.batches.create(
    input_file_id=batch_file.id,
    endpoint="/v1/chat/completions",
    completion_window="24h")

Print the batch job ID. Copy this value — it is needed to retrieve results if the kernel is restarted before the job completes.

In [ ]:
print(batch_job.id)

Store the batch job ID in a variable for use in the polling and download cells.

In [ ]:
bj_id = batch_job.id

## 1.2.5. Retrieve translation

Poll the batch job status. Run this cell repeatedly until `status` shows `completed`. Typical processing time is a few minutes to a few hours. Only proceed to the download cell once status is `completed`.

In [ ]:
# %% 4 ── poll until finished ─────────────────────────────────────────
batch = openai.batches.retrieve(bj_id)
print(batch.status)

Download the completed batch output file from OpenAI and save it locally as `batch_output.jsonl`. Each line contains the model response for one skill with the translated fields.

In [ ]:
# %% 5 ── download the translated output file ─────────────────────────
out_path = pathlib.Path("batch_output.jsonl")
content = openai.files.retrieve_content(batch.output_file_id)
with out_path.open('wb') as f:
    f.write(content.encode())
print("✔ downloaded", out_path)

## 1.2.6. Combine translation to output file

Parse the downloaded batch output and merge Russian translations into a CSV. Reads each line of `batch_output.jsonl`, extracts the translated fields from the function-call arguments, and builds a lookup dictionary keyed by `conceptUri`. The merged result is saved as `skills_ru.csv` — consumed by Stage 2 for all Russian-language job records.

In [15]:
# %% 6 ── merge English CSV with Russian translations ─────────────────
import json

out_path = "data/stage0/batch_output.jsonl"
ru_map = {}
with open(out_path, encoding="utf-8") as f:
    for line in f:
        args = json.loads(line)["response"]["body"]["choices"][0]["message"]["function_call"]["arguments"]
        inner = json.loads(args)
        uri = inner["conceptUri"]
        ru_map[uri] = {
            "preferredLabel": inner.get("preferredLabel_ru", ""),
            "altLabels": inner.get("altLabels_ru", []).split('|'),
            "description": inner.get("description_ru", "")
        }

---
© 2026 Yurii Kleban, Britta Rude. All rights reserved.